In [0]:
import importlib
import utils.storage_config as storage_config
from pyspark.sql import functions as F
importlib.reload(storage_config)
storage_config.spark = spark
storage_config.configure_storage()

In [0]:
df_customers = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("abfss://raw-data@secondstorage89.dfs.core.windows.net/customers/")
)

In [0]:
df = (
    df_customers
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

In [0]:
df.show(10)

In [0]:
df.printSchema()

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://bronze@secondstorage89.dfs.core.windows.net/customers/")